# Notebook 1: Data Understanding & Preprocessing
**Progetto:** Premier League Event Data 2020-21  
**Input:** `data/dataset.parquet` (dataset grezzo da Kaggle)  
**Output:** `data/processed/dataset_clean.parquet` (Silver Layer per Feature_extraction_EDA.ipynb)

---

## Obiettivo
In assenza di un Data Dictionary ufficiale, la fase iniziale di questo notebook si concentra sull'esplorazione strutturale e semantica del dataset (Premier League 2020-21).


### Pipeline di questo notebook
1. Data Ingestion e Analisi Strutturale
2. Data Understanding
3. Data Cleaning
4. Salvataggio Silver Layer

<a href="https://colab.research.google.com/github/Frestka/Mining_Premier_League/blob/main/notebooks/Data_Cleaning_and_Understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup e Caricamento

In [ ]:
# Rilevamento automatico ambiente
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys
    from pathlib import Path

    REPO_NAME = "Mining_Premier_League"
    REPO_URL  = "https://github.com/Frestka/Mining_Premier_League.git"

    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone -q {REPO_URL}
    os.chdir(REPO_NAME)
    sys.path.insert(0, os.path.abspath("."))

    !git fetch -q origin
    !git checkout -q origin/main
    !pip install -q -r requirements.txt
    Path("data/processed").mkdir(parents=True, exist_ok=True)

    import gdown
    FILES = {
        "data/dataset.parquet": "1fSuDQalfh2_EANL5sulWyucHCo81xZQZ",  # Bronze layer
        "data/20-21_plEventsData.csv":  "1inDdK5hzYSGRfUxbCo9tCoeKLpkHCPSR",  # CSV originale
    }

    for path, file_id in FILES.items():
        if not os.path.exists(path):
            print(f"Download {path}...")
            gdown.download(id=file_id, output=path, quiet=False, fuzzy=True)
        else:
            print(f"Gia presente, skip -> {path}")

    print(f"Setup Colab completato. Cartella: {os.getcwd()}")

else:
    print("Ambiente locale rilevato: setup Colab saltato automaticamente.")

In [ ]:
import sys, os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def setup_project():
    root = Path.cwd()
    while not (root / 'requirements.txt').exists() and root.parent != root:
        root = root.parent
    if not (root / 'requirements.txt').exists():
        raise RuntimeError(
            "File requirements.txt non trovato. Su Colab esegui prima la cella Setup Colab. "
            "In locale apri il notebook dalla root del repo o da notebooks/."
        )
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    os.chdir(root)
    return root

PROJECT_ROOT = setup_project()
print(f'Project root: {PROJECT_ROOT}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# In locale: se il parquet non esiste ma c'è il CSV, lo converte automaticamente
if not Path('data/dataset.parquet').exists() and Path('data/20-21_plEventsData.csv').exists():
    print("Converto CSV in Parquet per ottimizzare spazio e velocità...")
    pd.read_csv('data/20-21_plEventsData.csv').to_parquet('data/dataset.parquet', engine='pyarrow')
    print("Conversione completata!")

df = pd.read_parquet('data/dataset.parquet')
print(f'Dataset Bronze caricato con successo!')
print(f'Dimensioni: {df.shape[0]:,} righe x {df.shape[1]} colonne')

## Data Understanding

Osserviamo i dati grezzi: tipi, valori nulli, distribuzione degli eventi.

In [ ]:
# Struttura generale
print("--- STRUTTURA DEL DATASET ---")
df.info()

In [ ]:
# Analisi valori nulli
print("--- PERCENTUALE VALORI NULLI PER COLONNA ---")
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].round(2).astype(str) + '%')

In [ ]:
# Distribuzione tipi di evento e periodi
print("--- TOP 15 TIPI DI EVENTO ---")
print(df['type_displayName'].value_counts().head(15))

print("\n--- DISTRIBUZIONE PERIODI DI GIOCO ---")
print(df[['period_value', 'period_displayName']].value_counts().sort_index())

## Data Understanding

Di seguito il significato di ogni variabile, ricostruito interrogando i valori concreti.

### Identificatori e struttura
- **`Unnamed: 0`** (`srno`): Indice intra-partita che si resetta a 0 ad ogni nuovo match. Usato per costruire il `match_id`. Non ha nome perché è l'index di un DataFrame esportato senza rinominarlo.
- **`id`**: Identificatore univoco dell'evento, eccezioni gestite nella sezione successiva.
- **`eventId`**: ID alternativo con **due contatori paralleli**: alcune squadre usano 1,2,3...; altre 1.000.002, 1.000.003... per evitare collisioni, non è utilizzabile per identificare partite.
- **`match_id`**: Colonna creata per identificare i match.

### Temporali
- **`minute`**: Minuto dell'arbitro — si ferma a 45 e 90. Un gol al 47' sarà scritto come 45.
- **`second`**: Secondi dell'evento.
- **`expandedMinute`**: Minuto REALE incluso recupero.
- **`period_value` / `period_displayName`**: Fase di gioco. 1=FirstHalf, 2=SecondHalf, 14=PostGame, 16=PreMatch.
- **`minute == 32767`**: Valore sentinella rimosso — placeholder per eventi fuori dal tempo di gioco.

### Squadra e giocatore
- **`teamId`**: ID numerico squadra, 20 valori unici = 20 squadre PL.
- **`playerId` / `name` / `position` / `shirtNo`**: Dati del giocatore. Null per eventi di sistema (Start, End, FormationSet) dove non c'è un giocatore specifico → **Null strutturale**.
- **`position`**: 17 ruoli (GK, DR, DL, DC, DMC, MC, AMC, AML, AMR, MR, ML, FWL, FWR, FW, Sub...).
- **`shirtNo`**: Numero di maglia in formato float (Pandas forza float quando ci sono NaN in colonne int).

### Evento
- **`type_value` / `type_displayName`**: Tipo di evento (39 tipi: Pass, Tackle, Goal, Foul...).
- **`outcomeType_value` / `outcomeType_displayName`**: Esito — 1/Successful o 0/Unsuccessful.
- **`isTouch`**: True se c'è stato un contatto con la palla. False = evento amministrativo (fischio, sostituzione).

### Coordinate
- **`x` / `y`**: Posizione di inizio evento sul campo — scala **0-100** (percentuale, non metri).
- **`endX` / `endY`**: Posizione di arrivo. Null per eventi senza destinazione (tackle, foul) → **Null strutturale**.

### Specifiche per tiri e gol
- **`isShot` / `isGoal` / `isOwnGoal`**: True solo sull'evento corrispondente, altrimenti NaN → **Null strutturale** .
- **`goalMouthY` / `goalMouthZ`**: Coordinate porta dove il tiro ha raggiunto la linea. Null se non è un tiro → **Null strutturale**.
- **`blockedX` / `blockedY`**: Coordinate del blocco. Null se il tiro non è stato murato → **Null strutturale**.

### Cartellini
- **`cardType_value` / `cardType_displayName`**: 31=Yellow, 32=SecondYellow, 33=Red. Null se non è un evento Card → **Null strutturale**.

### Relazioni tra eventi
- **`relatedEventId`**: ID dell'evento collegato (es. il passaggio che ha generato il gol → utile per calcolare gli assist).
- **`relatedPlayerId`**: ID del giocatore correlato (es. in un fallo, chi lo subisce).

- **`qualifiers`**: json con molte informazioni aggregate.
- **`satisfiedEventsTypes`**: Array di ID numerici interni.

---
### Interpretazione dei Null

La maggior parte dei null in questo dataset è **strutturale**: il provider registra un valore solo se l'evento lo richiede. `goalMouthZ` è null perché il 98% degli eventi non è un tiro. `cardType` è null perché il 99% degli eventi non è un cartellino. Questi null **non vanno imputati** — NaN qui è l'informazione corretta.



### Identificazione delle Partite — Costruzione di `match_id`

Il dataset grezzo non ha una colonna `match_id`.  che viene creato da `Unnamed: 0` (poi rinominata srno): questo indice si resetta a **0 esattamente ad ogni fischio d'inizio** di una nuova partita. Una `cumsum()` su questa condizione produce un identificatore progressivo univoco per partita.

In [ ]:
# Rinominiamo
df = df.rename(columns={'Unnamed: 0': 'srno'})

# Ordinamento
df = df.sort_index()

# Costruzione match_id: ogni reset a 0 = nuova partita
df['match_id'] = (df['srno'] == 0).cumsum()

n_matches = df['match_id'].nunique()
print(f"Partite identificate: {n_matches}  (attese: 380 per PL 2020-21)")

# Verifica: ogni partita deve avere esattamente 2 squadre
teams_per_match = df.groupby('match_id')['teamId'].nunique()
print(f"Partite con esattamente 2 squadre: {(teams_per_match == 2).sum()} / {n_matches}")

## Data Cleaning

Applichiamo le operazioni di pulizia in ordine logico:

1. **Gestione duplicati su `id`**: 80 righe condividono lo stesso `id` però sono associati ad eventi diversi. Si tratta di azioni valide → **manteniamo le righe**.
2. **Trattamento `qualifiers`**: tramite parsing vettorializzato abbiamo estratto le informazioni utili — booleani (is_keypass, is_longball...), numerici (pass_length, pass_angle...) e categorici (zone, body_part). Una volta create queste ~28 nuove colonne, il JSON grezzo viene rimosso perchè ridondante.
**Trattamento `satisfiedEventsTypes`:** array di ID numerici interni di Opta (sotto-categorie dell'evento). Viene rimosso, perchè in assenza di un Data Dictionary ufficiale non è possibile decodificare questi codici in modo affidabile.
3. **Filtro fasi di gioco**: manteniamo `period_value` in [1,2,3,4,5] — escludiamo PreMatch (16) e PostGame (14).
4. **Rimozione sentinella temporale**: `minute == 32767` è un placeholder di sistema, non un evento reale.
5. **Casting booleani**: `isShot`, `isGoal`, `isOwnGoal` sono di tipo `object` (contengono NaN o stringa `'True'`). Convertiamo: `fillna(False)` trasforma NaN → False booleano; `astype(bool)` trasforma `'True'` → True.
6. **Correzione `shirtNo`**: da float a int (Pandas forza float quando ci sono NaN in colonne intere).

In [ ]:
import pandas as pd
import numpy as np

print(f"Dimensioni prima del cleaning: {df.shape[0]:,} righe × {df.shape[1]} colonne")

# 1. Duplicati su id
n_duplicati = df['id'].duplicated().sum()
print(f"\nRighe con 'id' duplicato: {n_duplicati} → MANTENUTE")

# ==============================================================================
# 2. ESTRAZIONE QUALIFIERS (METRICHE TATTICHE AVANZATE
# ==============================================================================
print("\nEstrazione feature avanzate dai qualifiers in corso...")

BOOL_QUALIFIERS = [
    'Longball', 'Chipped', 'Cross', 'HeadPass', 'Throughball',
    'KeyPass', 'BigChance', 'BigChanceCreated', 'Assisted', 'IntentionalAssist',
    'FastBreak', 'LayOff', 'FromCorner', 'SetPiece',
    'FreekickTaken', 'CornerTaken', 'Penalty', 'ThrowIn',
    'Blocked', 'OutfielderBlock', 'SixYardBlock', 'PlayerCaughtOffside'
]

# Estrazione Booleani
for q in BOOL_QUALIFIERS:
    col_name = 'is_' + q.lower()
    df[col_name] = df['qualifiers'].str.contains(f"'displayName': '{q}'", na=False)

# Estrazione Numerici
df['pass_length'] = df['qualifiers'].str.extract(r"'displayName': 'Length'\}, 'value': '([^']+)'").astype(float)
df['pass_angle'] = df['qualifiers'].str.extract(r"'displayName': 'Angle'\}, 'value': '([^']+)'").astype(float)
df['goal_mouth_y'] = df['qualifiers'].str.extract(r"'displayName': 'GoalMouthY'\}, 'value': '([^']+)'").astype(float)
df['goal_mouth_z'] = df['qualifiers'].str.extract(r"'displayName': 'GoalMouthZ'\}, 'value': '([^']+)'").astype(float)

# Estrazione Categorici
df['zone'] = df['qualifiers'].str.extract(r"'displayName': 'Zone'\}, 'value': '([^']+)'")

# Estrazione Body Part
df['body_part'] = None
for bp in ['RightFoot', 'LeftFoot', 'Head', 'OtherBodyPart']:
    df.loc[df['qualifiers'].str.contains(f"'displayName': '{bp}'", na=False), 'body_part'] = bp

# --- SANITY CHECKS ---
print("\n--- SANITY CHECKS ESTRAZIONE ---")
keypasses = df['is_keypass'].sum()
print(f"KeyPass trovati: {keypasses}")
print("Distribuzione Body Part:")
print(df['body_part'].value_counts().to_string())
# ---------------------

# 3. Rimozione colonne
colonne_da_scartare = ['qualifiers', 'satisfiedEventsTypes']
df_clean = df.drop(columns=[c for c in colonne_da_scartare if c in df.columns])
print(f"\nColonne grezze rimosse per pulizia: {colonne_da_scartare}")

# 4. Filtro fasi di gioco
n_before = len(df_clean)
df_clean = df_clean[df_clean['period_value'].isin([1, 2, 3, 4, 5])]
print(f"\nEventi pre/post-match rimossi: {n_before - len(df_clean):,}")

# 5. Rimozione sentinella
n_before = len(df_clean)
df_clean = df_clean[df_clean['minute'] != 32767]
print(f"Righe con minute=32767 rimosse: {n_before - len(df_clean):,}")

# 6. Casting booleani
bool_cols_native = ['isShot', 'isGoal', 'isOwnGoal']
bool_cols_qualifier = ['is_' + q.lower() for q in BOOL_QUALIFIERS]

for col in bool_cols_native:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(False).astype(bool)

for col in bool_cols_qualifier:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(bool)

print("\nCasting booleani applicato.")

# 7. shirtNo
if 'shirtNo' in df_clean.columns:
    df_clean['shirtNo'] = df_clean['shirtNo'].fillna(-1).astype(int)

print(f"\nDimensioni dopo il cleaning: {df_clean.shape[0]:,} righe × {df_clean.shape[1]} colonne")


In [ ]:
# Verifica finale del dataset pulito
print("--- VERIFICA FINALE ---")
print(f"Partite nel dataset pulito: {df_clean['match_id'].nunique()}")
print(f"Squadre uniche: {df_clean['teamId'].nunique()}")
print(f"Periodi rimasti: {df_clean['period_displayName'].unique()}")

bool_cols = bool_cols_native + bool_cols_qualifier
bool_cols_presenti = [c for c in bool_cols if c in df_clean.columns]

print(f"Tipi booleani: {df_clean[bool_cols_presenti].dtypes.to_dict()}")
print(f"\nNull residui (strutturali attesi):")
null_residui = df_clean.isnull().sum()
print(null_residui[null_residui > 0])

## Salvataggio — Silver Layer

Il dataset viene salvato in formato Parquet (compressione snappy) per massima efficienza in lettura.

I null residui visibili sopra sono tutti **strutturali** e non richiedono imputazione: verranno gestiti naturalmente dall'aggregazione per partita nel Notebook 2, dove la funzione `extract_features` usa `safe_rate()` per evitare divisioni per zero e `.mean()` che ignora i NaN per definizione.

In [ ]:
# Salvataggio Silver Layer
Path('data/processed').mkdir(parents=True, exist_ok=True)
df_clean.to_parquet('data/processed/dataset_clean.parquet', engine='pyarrow', compression='snappy')

print(f'Dataset Silver salvato correttamente.')
print(f'  Percorso: data/processed/dataset_clean.parquet')
print(f'  Dimensioni: {df_clean.shape[0]:,} righe x {df_clean.shape[1]} colonne')
print(f'  Colonne: {list(df_clean.columns)}')